# P06 — Перенос обучения и ML системы / Transfer learning and ML systems

**Четыре блока / Four blocks:** материал → пример кода → ваше задание → автоматическая проверка. / Explanation → worked code example → your coding task → automatic check.

Выполняйте последовательно. В каждом задании замените `raise NotImplementedError` своим кодом, запустите проверку и затем продолжайте. Пустой шаблон намеренно останавливается на незавершённом задании. / Work in order. Replace each `raise NotImplementedError`, run its check, then continue. The starter intentionally stops at an unfinished task.

Проверки не выставляют оценку за практику. После каждого задания есть короткий письменный ответ; его обсуждаем в группе. / Checks do not assign a practice grade. Discuss the short written response after each exercise.


## Данные и протокол / Data and protocol
**CIFAR-10**: реальные цветные изображения 32×32, 10 классов. Объект — одно изображение; признаки — RGB-пиксели, доступные при запросе; цель — категория объекта. / Real 32×32 colour images in 10 classes. One observation is one image; RGB pixels are available at prediction time; the target is the object category.
Источник / Source: https://www.cs.toronto.edu/~kriz/cifar.html (Krizhevsky, 2009). Официальные части: 50 000 train, 10 000 test. / Official partitions: 50,000 train and 10,000 test.
Train и Validation берём без пересечений из официального train; Test — отдельная фиксированная подвыборка official test. Метки Test используются только для фиксированного стратифицированного отбора и финальной оценки. / Train and Validation are disjoint subsets of official train; Test is a fixed stratified subset of official test. Test labels are used only for predefined stratified sampling and final scoring.
Нет ID источника/автора или времени: нельзя утверждать перенос на новых авторов либо будущие данные. Маленькая учебная подвыборка даёт шумную оценку, не результат на полном benchmark. Возможное пересечение источников ImageNet и CIFAR здесь не проверено. / No source/writer/time IDs: this does not establish generalization to unseen authors or future data. Small-subset scores are noisy, not full-benchmark results. Source overlap between ImageNet and CIFAR has not been audited here.
Первый запуск скачивает CIFAR-10 (~163 MiB) и веса (~10 MiB). Интернет нужен для скачивания; платного API нет. CPU достаточно; извлечение признаков может занять несколько минут. / First run downloads data (~163 MiB) and weights (~10 MiB). Internet is needed for downloads; no paid API. CPU is sufficient; feature extraction may take several minutes.


## Подготовка / Setup
Colab обычно содержит зависимости. При ошибке импорта установите совместимые `torch` и `torchvision` по инструкции PyTorch, а также numpy, pandas, matplotlib, scikit-learn и pillow; затем перезапустите сеанс. / Colab normally includes dependencies. If imports fail, install a compatible torch/torchvision pair following PyTorch instructions plus numpy, pandas, matplotlib, scikit-learn and pillow, then restart the runtime.
https://pytorch.org/get-started/locally/


In [ ]:
import json, time, platform # Журнал, таймер и версия Python / Log, timer and Python version.
from pathlib import Path # Сохранение журнала / Save the run log.
import numpy as np # Индексы и признаки / Indices and features.
import pandas as pd # Таблицы сравнения / Comparison tables.
import matplotlib.pyplot as plt # Изображения и матрица ошибок / Images and confusion matrix.
from PIL import Image # Представление RGB для transform / RGB representation for transforms.
import torch, torchvision, sklearn # Библиотеки и реальные версии / Libraries and actual versions.
from torch import nn # Замена классификатора / Replace the classifier.
from torch.utils.data import Dataset, DataLoader # Упорядоченные батчи / Ordered batches.
from torchvision.datasets import CIFAR10 # Реальные изображения / Real images.
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights # Модель и фиксированные веса / Model and fixed weights.
from sklearn.pipeline import make_pipeline # Связать scaler и head / Combine scaler and head.
from sklearn.preprocessing import StandardScaler # Стандартизация по Train / Training-only scaling.
from sklearn.linear_model import LogisticRegression # Обучаемая голова / Trainable head.
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay # Метрики / Metrics.
SEED = 2026 # Фиксированный протокол / Fixed protocol.
torch.set_num_threads(2) # Ограничить потоки CPU / Limit CPU threads.
checks_passed = set() # Отметить пройденные проверки / Record passed checks.
print({'python': platform.python_version(), 'torch': torch.__version__, 'torchvision': torchvision.__version__, 'sklearn': sklearn.__version__}) # Версии запуска / Runtime versions.
official_train = CIFAR10('data', train=True, download=True) # Официальный Train / Official training source.
official_test = CIFAR10('data', train=False, download=True) # Независимый источник Test / Separate official test source.
classes = list(official_train.classes) # Имена десяти классов / Ten class names.
y_source = np.asarray(official_train.targets) # Метки источника Train / Training-source labels.
y_test_source = np.asarray(official_test.targets) # Метки только для фиксированного отбора Test / Labels for prespecified test sampling.


## Блок 1 — Данные и разбиение / Block 1 — Data and splits
### Материал / Explanation
Индекс имеет смысл вместе с источником: `train:42` и `test:42` — разные изображения. Train/Validation используют один источник и не должны пересекаться. Число объектов класса — сумма индикаторов, доля — число / размер части. / An index must be paired with its source: train:42 and test:42 are different images. Train/Validation share a source and must be disjoint. A class count sums membership indicators; its fraction divides the count by partition size.

Разбиение фиксируем заранее: по 60/20/20 объектов каждого класса, всего 600/200/200. / Fix 60/20/20 per class, totaling 600/200/200.
### Пример кода / Worked example
Отбираем части и считаем число объектов одного класса в Train. / Select partitions and count one training class.


In [ ]:
def split_indices(train_labels, test_labels, seed): # Выбрать фиксированные поднаборы / Select fixed subsets.
    rng = np.random.default_rng(seed) # Воспроизводимое перемешивание / Reproducible shuffle.
    tr, va, te = [], [], [] # Три списка индексов / Three index lists.
    for label in range(10): # Сохранить все классы / Retain all classes.
        ids = rng.permutation(np.flatnonzero(train_labels == label)) # Перемешать Train ID класса / Shuffle training class IDs.
        tids = rng.permutation(np.flatnonzero(test_labels == label)) # Перемешать Test ID класса / Shuffle test class IDs.
        tr.extend(ids[:60]) # Первые 60 в Train / First 60 into Train.
        va.extend(ids[60:80]) # Следующие 20 в Validation / Next 20 into Validation.
        te.extend(tids[:20]) # 20 из другого источника / Twenty from the other source.
    return tuple(np.asarray(ids, dtype=np.int64) for ids in (tr, va, te)) # Вернуть массивы / Return arrays.
train_ids, val_ids, test_ids = split_indices(y_source, y_test_source, SEED) # Заморозить разбиение / Freeze partitions.
assert not set(train_ids) & set(val_ids) # Проверить непересечение одного источника / Check disjointness within one source.
y_train, y_val = y_source[train_ids], y_source[val_ids] # Train и Validation метки / Training and validation labels.
print(classes[0], int(np.count_nonzero(y_train == 0)), len(y_train)) # Один класс как пример / One class as an example.


### Задание 1 / Exercise 1
Реализуйте `summarize_split(labels, ids)`: вход — 1D массив всех меток источника и непустые допустимые индексы без повторов. Верните словарь `n` (Python int), `counts` (целочисленный NumPy массив длины 10), `fractions` (float массив длины 10). Учитывайте отсутствующие классы нулями, не меняйте входы. Не прописывайте 60/20 вручную. / Return n as a Python int, counts as a length-10 integer NumPy array and fractions as a length-10 float array. Input is a 1D source-label array and nonempty valid unique indices. Include absent classes as zero; preserve inputs and do not hardcode 60/20.

После проверки выведите таблицу всех трёх частей. Объясните в 2–3 предложениях, почему числовое пересечение Train и Test ID допустимо, а Train и Validation — нет. / After checking, display all three partitions and explain why numerical Train/Test ID overlap is allowed but Train/Validation overlap is not.


In [ ]:
def summarize_split(labels, ids): # Описать выбранную часть / Describe the selected partition.
    raise NotImplementedError('Задание 1 / Exercise 1') # Напишите решение / Write your solution.


In [ ]:
for source, ids in [(y_source, train_ids), (y_source, val_ids), (y_test_source, test_ids), (y_source, val_ids[:7][::-1])]: # Реальные части и неполный срез / Real partitions and a partial slice.
    old_labels, old_ids = source.copy(), ids.copy() # Снимок входов / Snapshot inputs.
    summary = summarize_split(source, ids) # Вызвать решение / Call the solution.
    expected = np.array([np.count_nonzero(source[ids] == k) for k in range(10)]) # Независимый подсчёт / Independent counting.
    assert type(summary['n']) is int and summary['n'] == len(ids) # Проверить размер / Check size.
    assert isinstance(summary['counts'], np.ndarray) and summary['counts'].dtype.kind in 'iu' # Целочисленный массив / Integer array.
    assert summary['counts'].shape == (10,) and np.array_equal(summary['counts'], expected) # Все классы / All classes.
    assert isinstance(summary['fractions'], np.ndarray) and summary['fractions'].shape == (10,) and summary['fractions'].dtype.kind == 'f' # Массив долей / Fraction array.
    assert np.allclose(summary['fractions'], expected / len(ids)) # Правильные доли / Correct fractions.
    assert np.array_equal(source, old_labels) and np.array_equal(ids, old_ids) # Входы неизменны / Inputs unchanged.
checks_passed.add(1) # Зафиксировать успех / Record success.
display(pd.DataFrame({name: summarize_split(source, ids)['counts'] for name, source, ids in [('Train', y_source, train_ids), ('Validation', y_source, val_ids), ('Test', y_test_source, test_ids)]}, index=classes)) # Таблица классов / Class-count table.
print('Задание 1 OK / Exercise 1 OK') # Результат самопроверки / Self-check result.


**Ваше объяснение / Your explanation:** …

## Блок 2 — Замороженные признаки / Block 2 — Frozen features
### Материал / Explanation
MobileNetV3 уже обучен на ImageNet. Удаляем его старый classifier: получаем 576 признаков, а не CIFAR-метку. `requires_grad_(False)` запрещает градиенты параметров; `eval()` отключает dropout и обновление статистик BatchNorm; `inference_mode()` отключает построение графа. Эти действия имеют разные роли. / Remove the pretrained ImageNet classifier to obtain 576 features, not a CIFAR label. requires_grad_(False), eval() and inference_mode() disable parameter gradients, training behavior and graph recording respectively; their roles differ.

Порядок строк признаков должен совпадать с порядком ID и меток. / Feature rows must retain ID and label order.
### Пример кода / Worked example
Один проход одного настоящего изображения. / One forward pass for one real image.


In [ ]:
weights = MobileNet_V3_Small_Weights.IMAGENET1K_V1 # Явно выбрать веса / Explicit weights version.
preprocess = weights.transforms() # Нормировка и размер для этих весов / Matching normalization and resize.
backbone = mobilenet_v3_small(weights=weights) # Загрузить предобученную сеть / Load pretrained network.
backbone.classifier = nn.Identity() # Оставить 576 признаков / Retain 576 features.
backbone.requires_grad_(False) # Не обучать параметры / Freeze parameters.
backbone.eval() # Выключить поведение обучения / Disable training behavior.
example_image = official_train[int(val_ids[0])][0] # Реальное изображение Validation / Actual validation image.
with torch.inference_mode(): # Не строить граф / Disable graph recording.
    example_features = backbone(preprocess(example_image).unsqueeze(0)).cpu().numpy() # Один батч / Single-image batch.
print(example_features.shape) # Ожидается (1, 576) / Expect (1, 576).


### Задание 2 / Exercise 2
Реализуйте `extract_features(dataset, ids, model, transform, batch_size=32)`. Вход: непустой массив ID, CPU-модель и transform. Создайте Dataset выбранных изображений, DataLoader с `shuffle=False`, `num_workers=0`; вызовите `model.eval()`, обработайте батчи внутри `torch.inference_mode()`. Верните конечный NumPy массив `(len(ids),576)` в исходном порядке. Параметры и буферы модели менять нельзя. / Build a selected-image Dataset and DataLoader with shuffle=False and num_workers=0; use eval() and inference_mode() on CPU. Return a finite NumPy array (len(ids),576) in input order without modifying model parameters or buffers.

Проверка использует реальный обратный порядок ID, неполный последний батч и независимые одиночные проходы. Объясните, как shuffle=True без соответствующего изменения меток испортит обучение головы. / The check uses reversed real IDs, an incomplete last batch and independent single-image forwards. Explain how shuffling without matching label order corrupts head training.


In [ ]:
def extract_features(dataset, ids, model, transform, batch_size=32): # Извлечь признаки / Extract features.
    raise NotImplementedError('Задание 2 / Exercise 2') # Напишите решение / Write your solution.


In [ ]:
probe_ids = val_ids[:5][::-1] # Нетривиальный порядок настоящих ID / Nontrivial order of real IDs.
state_before = {k: v.clone() for k, v in backbone.state_dict().items()} # Сохранить параметры и буферы / Snapshot parameters and buffers.
backbone.train() # Проверить, что функция сама включает eval / Check that the function enables eval.
probe = extract_features(official_train, probe_ids, backbone, preprocess, batch_size=2) # Последний батч неполный / Incomplete final batch.
assert isinstance(probe, np.ndarray) and probe.shape == (5, 576) and np.isfinite(probe).all() # Проверить контракт / Check contract.
assert not backbone.training and all(not p.requires_grad for p in backbone.parameters()) # Проверить режим / Check mode.
assert all(torch.equal(state_before[k], v) for k, v in backbone.state_dict().items()) # Параметры и буферы прежние / State unchanged.
with torch.inference_mode(): # Независимый эталон без графа / Independent reference without graph.
    reference = np.vstack([backbone(preprocess(official_train[int(i)][0]).unsqueeze(0)).cpu().numpy() for i in probe_ids]) # Одиночные проходы / Single-image forwards.
assert np.allclose(probe, reference, atol=1e-5, rtol=1e-4) # Числа и порядок совпадают / Values and order match.
checks_passed.add(2) # Зафиксировать успех / Record success.
start = time.perf_counter() # Начало измерения / Start timing.
F_train = extract_features(official_train, train_ids, backbone, preprocess) # Признаки Train / Training features.
F_val = extract_features(official_train, val_ids, backbone, preprocess) # Признаки Validation / Validation features.
feature_seconds = time.perf_counter() - start # Полное время извлечения / Total extraction time.
print('Задание 2 OK / Exercise 2 OK', F_train.shape, F_val.shape, feature_seconds) # Отчёт / Report.


**Ваше объяснение / Your explanation:** …

## Блок 3 — Выбор по Validation / Block 3 — Validation selection
### Материал / Explanation
Обучаем только линейную голову. Scaler обучается внутри Pipeline только на Train. Сравниваем фиксированные C=0.01,0.1,1.0 по Validation macro-F1; точная ничья → меньший C. Test пока не используем. / Train only the linear head. The pipeline fits the scaler on Train only. Compare fixed C values by validation macro-F1; exact ties select smaller C. Do not use Test yet.
### Пример кода / Worked example
Обучаем пиксельный baseline и три головы, сохраняем их Validation-результаты. Выбор победителя — ваше задание. / Train the pixel baseline and three heads; record validation scores. Selecting the winner is your task.


In [ ]:
def fit_head(features, labels, C): # Обучить только голову / Fit only the head.
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000, random_state=SEED)).fit(features, labels) # Train-only Pipeline / Training-only pipeline.
def pixels(dataset, ids): # Пиксельный baseline / Pixel baseline.
    return dataset.data[ids].reshape(len(ids), -1).astype(np.float32) / 255.0 # Развернуть RGB / Flatten RGB.
def report(y, pred): # Две метрики / Two metrics.
    return {'accuracy': float(accuracy_score(y, pred)), 'macro_f1': float(f1_score(y, pred, average='macro', zero_division=0))} # Отчёт / Metric report.
baseline = fit_head(pixels(official_train, train_ids), y_train, 0.1) # Фиксированный baseline / Fixed baseline.
baseline_pred = baseline.predict(pixels(official_train, val_ids)) # Только Validation / Validation only.
heads, records = {}, [] # Модели и результаты / Models and scores.
for C in [0.01, 0.1, 1.0]: # Заранее заданные кандидаты / Prespecified candidates.
    heads[C] = fit_head(F_train, y_train, C) # Обучить голову на Train / Fit head on Train.
    score = report(y_val, heads[C].predict(F_val))['macro_f1'] # Оценить Validation / Evaluate validation.
    records.append({'C': C, 'val_macro_f1': score}) # Сохранить полную точность / Preserve full precision.
display(pd.DataFrame(records)) # Показать сравнение / Display comparison.
print('Pixel baseline:', report(y_val, baseline_pred)) # Показать baseline / Show baseline.


### Задание 3 / Exercise 3
Реализуйте `choose_run(records)`: непустой список словарей с положительным C и конечным val_macro_f1. Верните C максимального macro-F1; при точном равенстве меньший C. Не изменяйте список и записи, не округляйте метрики. / Return the C with maximum validation macro-F1, breaking exact ties by smaller C. Preserve the nonempty input list and records; do not round scores.

После проверки найдите в показанном Validation-разборе реальную ошибку: ID, изображение, true, transfer и baseline. Запишите наблюдение и отдельно гипотезу причины; если ошибок нет, так и напишите. Объясните выбранный C числами. / After checking, inspect an actual validation failure: ID, image, reference, transfer and baseline predictions. Separate observation from a possible cause; report honestly if there are no errors. Explain selection numerically.


In [ ]:
def choose_run(records): # Выбрать гиперпараметр / Select the hyperparameter.
    raise NotImplementedError('Задание 3 / Exercise 3') # Напишите решение / Write your solution.


In [ ]:
original = json.dumps(records, sort_keys=True) # Защитить исходные записи / Snapshot original records.
maximum = max(row['val_macro_f1'] for row in records) # Лучший реальный score / Best actual score.
expected = min(row['C'] for row in records if row['val_macro_f1'] == maximum) # Эталон по контракту / Contract reference.
for case in [records, list(reversed(records)), records[:1], records[-1:], [{'C': a['C'], 'val_macro_f1': b['val_macro_f1']} for a, b in zip(records, records[::-1])]]: # Порядок и один кандидат / Order and singleton.
    saved = json.dumps(case, sort_keys=True) # Снимок случая / Snapshot case.
    best_score = max(row['val_macro_f1'] for row in case) # Лучшее значение / Best value.
    assert choose_run(case) == min(row['C'] for row in case if row['val_macro_f1'] == best_score) # Проверить выбор / Check selection.
    assert json.dumps(case, sort_keys=True) == saved # Не менять вход / Preserve input.
tie = [{'C': row['C'], 'val_macro_f1': records[0]['val_macro_f1']} for row in records[::-1]] # Проверка ничьей с реальным score / Tie fixture using an actual score.
assert choose_run(tie) == min(row['C'] for row in tie) # Меньший C независимо от порядка / Smaller C regardless of order.
assert json.dumps(records, sort_keys=True) == original # Исходные записи прежние / Original records unchanged.
selected_C = choose_run(records) # Зафиксировать C до Test / Freeze C before Test.
head = heads[selected_C] # Получить выбранную голову / Retrieve selected head.
pred_val = head.predict(F_val) # Предсказать Validation / Predict validation.
errors = np.flatnonzero(pred_val != y_val) # Позиции реальных ошибок / Actual error positions.
print('Selected C:', selected_C, 'Validation errors:', len(errors), '/', len(y_val)) # Результат / Result.
ConfusionMatrixDisplay(confusion_matrix(y_val, pred_val, labels=np.arange(10)), display_labels=classes).plot(xticks_rotation=60) # Матрица Validation / Validation confusion matrix.
plt.show() # Показать график / Display plot.
if len(errors): # Не придумывать ошибку при её отсутствии / Do not invent errors.
    pos = int(errors[0]) # Первая доступная ошибка / First available error.
    plt.imshow(official_train.data[val_ids[pos]]) # Реальное изображение / Actual image.
    plt.axis('off') # Убрать оси / Hide axes.
    plt.show() # Показать пример / Display example.
    print({'id': int(val_ids[pos]), 'true': classes[y_val[pos]], 'transfer': classes[pred_val[pos]], 'baseline': classes[baseline_pred[pos]]}) # Сопоставить ответы / Compare predictions.
else: # Честно отметить отсутствие / Honestly report absence.
    print('Ошибок Validation нет / No validation errors') # Сообщить факт / Report fact.
checks_passed.add(3) # Зафиксировать успех / Record success.
print('Задание 3 OK / Exercise 3 OK') # Самопроверка / Self-check.


**Ваше объяснение и разбор ошибки / Your explanation and error analysis:** …

## Блок 4 — Функция предсказания / Block 4 — Prediction function
### Материал / Explanation
На входе сервиса фиксируем контракт: NumPy uint8 RGB, форма (32,32,3). Молча исправленный grayscale или float может означать иной диапазон/семантику. Явный отказ помогает найти ошибку источника. Порядок вероятностей соответствует `head.classes_`, а не произвольному словарю. / The service accepts a NumPy uint8 RGB array of shape (32,32,3). Silently converting grayscale or float data may hide a range/semantic mismatch. Reject invalid input explicitly. Probability columns follow head.classes_.

Задержка всей системы включает preprocess + backbone + head; время только head недостаточно. / End-to-end latency includes preprocessing, backbone and head, not just head inference.
### Пример кода / Worked example
Получаем вероятности уже извлечённых Validation-признаков и показываем соответствие колонок меткам. / Get probabilities for existing validation features and inspect column-to-label mapping.


In [ ]:
prob_example = head.predict_proba(F_val[:1])[0] # Вероятности одного объекта / One observation's probabilities.
print(list(zip(head.classes_.tolist(), prob_example.tolist()))) # Порядок колонок / Column order.
print('Input:', official_train.data[val_ids[0]].shape, official_train.data[val_ids[0]].dtype) # Контракт настоящего RGB / Actual RGB contract.


### Задание 4 / Exercise 4
Реализуйте `predict_image(image, model, transform, head, names)`. Для входа, не являющегося np.ndarray с dtype uint8 и формой (32,32,3), выбрасывайте ValueError **до запуска модели**. Для правильного входа: Image.fromarray → transform → batch dimension → eval/inference_mode → CPU NumPy → predict_proba. Верните `class_id` (Python int из head.classes_), `label` (names[class_id]), `probabilities` (словарь имя класса → Python float для всех классов). Не изменяйте изображение и модель. / Reject anything other than a uint8 NumPy RGB array of shape (32,32,3) with ValueError before model execution. Apply the stated inference pipeline and return class_id, label and a complete probability dictionary using head.classes_. Preserve the image and model.

Проверяем три реальные RGB-картинки, grayscale, float RGB и неправильный размер, полученные из реальной картинки. После проверки сравните три времени и объясните, почему они не обязаны складываться точно. / Check three actual RGB images plus grayscale, float RGB and wrong-size variants derived from a real image. Then compare the three timings and explain why they need not add exactly.


In [ ]:
def predict_image(image, model, transform, head, names): # Обработать запрос / Handle a request.
    raise NotImplementedError('Задание 4 / Exercise 4') # Напишите решение / Write your solution.


In [ ]:
snapshot = {k: v.clone() for k, v in backbone.state_dict().items()} # Снимок модели / Model snapshot.
for pos in [0, 11, 23]: # Разные реальные изображения / Different real images.
    image = official_train.data[val_ids[pos]].copy() # Копия входа / Input copy.
    before_image = image.copy() # Снимок для проверки / Snapshot for checking.
    result = predict_image(image, backbone, preprocess, head, classes) # Вызвать сервис / Call service.
    expected_p = head.predict_proba(F_val[pos:pos+1])[0] # Независимый batch-эталон / Independent batch reference.
    assert type(result['class_id']) is int and result['class_id'] == int(head.classes_[np.argmax(expected_p)]) # Правильная метка / Correct class.
    assert result['label'] == classes[result['class_id']] and set(result['probabilities']) == set(classes) # Имена и полнота / Names and completeness.
    actual_p = np.array([result['probabilities'][classes[int(k)]] for k in head.classes_]) # Сопоставить по меткам / Align by labels.
    assert all(type(v) is float for v in result['probabilities'].values()) # Сериализуемые числа / Serializable numbers.
    assert np.allclose(actual_p, expected_p, atol=1e-5, rtol=1e-4) and np.isclose(actual_p.sum(), 1) # Значения и сумма / Values and sum.
    assert np.array_equal(image, before_image) # Вход неизменён / Input unchanged.
image = official_train.data[val_ids[0]] # Источник неправильных форматов / Source for invalid variants.
class MustNotRun: # Страж ранней валидации / Guard for early validation.
    def eval(self): # Должен быть недостижим / Must not be reached.
        raise AssertionError('Validate before model / Проверяйте до модели') # Поймать позднюю проверку / Catch late validation.
for invalid in [image[:, :, 0], image.astype(np.float32), image[:16], image.tolist()]: # Нарушения контракта реального RGB / Contract violations derived from real RGB.
    try: # Ожидаем явный отказ / Expect explicit rejection.
        predict_image(invalid, MustNotRun(), preprocess, head, classes) # Проверить ранний отказ / Check early rejection.
    except ValueError: # Требуемый тип исключения / Required exception type.
        pass # Ожидаемое поведение / Expected behavior.
    else: # Неверный вход принят / Invalid input accepted.
        raise AssertionError('Expected ValueError / Ожидался ValueError') # Сообщить ошибку / Report failure.
assert all(torch.equal(snapshot[k], v) for k, v in backbone.state_dict().items()) # Модель прежняя / Model unchanged.
checks_passed.add(4) # Зафиксировать успех / Record success.
print('Задание 4 OK / Exercise 4 OK') # Самопроверка / Self-check.


### Применение и обсуждение / Application and discussion
Прогрейте функции, затем измерьте медиану 10 вызовов. Это локальный CPU без сети и очередей, не SLA сервиса. / Warm up, then measure the median of ten calls. This is local CPU timing without network or queues, not a service SLA.


In [ ]:
def measure_ms(call, repeats=10): # Измерить медианную задержку / Measure median latency.
    call() # Прогрев / Warm-up.
    durations = [] # Измерения / Measurements.
    for _ in range(repeats): # Повторить вызов / Repeat call.
        started = time.perf_counter() # Начать таймер / Start timer.
        call() # Выполнить операцию / Execute operation.
        durations.append((time.perf_counter() - started) * 1000) # Миллисекунды / Milliseconds.
    return float(np.median(durations)) # Устойчивая сводка / Median summary.
latency = {'features_ms': measure_ms(lambda: extract_features(official_train, val_ids[:1], backbone, preprocess, 1)), 'head_ms': measure_ms(lambda: head.predict_proba(F_val[:1])), 'request_ms': measure_ms(lambda: predict_image(image, backbone, preprocess, head, classes))} # Три области измерения / Three timing scopes.
print(latency) # Сравнить на одной машине / Compare on one machine.


**Ваше объяснение отказов и задержки / Your explanation of rejection and latency:** …

## Финальная проверка замороженного решения / Final frozen evaluation
Это завершение четырёх блоков, не пятое задание. Выполняется один раз после проверок. По Test не меняем C, метрику и split. / This closes the four blocks; it is not a fifth exercise. Run once after checks. Do not retune C, metrics or splits using Test.


In [ ]:
assert checks_passed == {1, 2, 3, 4} # Все задания проверены / All tasks checked.
selection = {'C': selected_C, 'criterion': 'validation macro-F1; exact tie -> smaller C', 'seed': SEED, 'weights': 'MobileNet_V3_Small_Weights.IMAGENET1K_V1'} # Заморозить протокол / Freeze protocol.
print('FROZEN BEFORE TEST:', selection) # Зафиксировать решение / Record decision.
F_test = extract_features(official_test, test_ids, backbone, preprocess) # Test только теперь / Test only now.
y_test = y_test_source[test_ids] # Метки для финальной оценки / Labels for final scoring.
test_results = {'pixels': report(y_test, baseline.predict(pixels(official_test, test_ids))), 'transfer': report(y_test, head.predict(F_test))} # Финальные метрики / Final metrics.
print('FINAL TEST:', test_results) # Вывести без нового выбора / Report without retuning.
experiment_log = {**selection, 'assignment': 'P06_Transfer_Learning', 'versions': {'torch': torch.__version__, 'torchvision': torchvision.__version__, 'sklearn': sklearn.__version__}, 'train_ids': train_ids.tolist(), 'val_ids': val_ids.tolist(), 'test_ids': test_ids.tolist(), 'validation_records': records, 'test_results': test_results, 'latency': latency, 'latency_scope': 'warm local CPU, batch=1, excludes network', 'preprocess': str(preprocess), 'class_names': classes} # Воспроизводимый журнал / Reproducible log.
Path('experiment_p06.json').write_text(json.dumps(experiment_log, ensure_ascii=False, indent=2)) # Сохранить журнал практики / Save practice log.
